# DataMate trên Colab Pro: toàn bộ pipeline với Qwen local

Chạy lần lượt từ trên xuống: clone repo → Qwen tạo manifest → ingestion toàn bộ train/public test → kiểm tra DuckDB → Qwen rewrite/sinh SQL → chấm train → xuất bài nộp.

Đầu vào là `dataset/train` và `dataset/public_test` trong repo clone. Không cần Gemini API key. Model được nạp một lần và dùng lại giữa các bước. Ingestion dùng Python/DuckDB, mọi tác vụ LLM dùng Qwen local. Manifest được sinh theo từng bảng nguồn, kiểm định schema, cột và bằng chứng; lỗi sẽ dừng bước chuẩn bị dữ liệu.

Chọn GPU trong **Runtime → Change runtime type**. `hf_quant="auto"` chọn lượng tử hóa theo VRAM; mức context vẫn cần kiểm tra thực tế. Kết quả mặc định ở `/content/datamate_sql_out`, bị mất khi Colab xóa runtime. Có thể đặt `OUT` trong Drive để giữ kết quả; khi đó notebook tự mount Drive.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import base64
import getpass
import os
import subprocess
import sys

@dataclass
class ColabRuntime:
    repo_url: str = "https://github.com/Oszumeiii/ZAIC_2026.git"
    branch: str = "main"
    checkout_dir: str = "/content/ZAIC_2026"

    def checkout(self):
        from google.colab import userdata
        try:
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = getpass.getpass("GitHub token cho repo private: ")
        if not token:
            raise RuntimeError("Thiếu GITHUB_TOKEN")
        auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        git = ["git", "-c", f"http.extraHeader=Authorization: Basic {auth}"]
        target = Path(self.checkout_dir)
        def run(args, cwd=None):
            result = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
            output = (result.stdout + result.stderr).replace(token, "***").replace(auth, "***")
            print(output[-1500:])
            if result.returncode:
                raise RuntimeError("Git lỗi: kiểm tra token, nhánh và thay đổi trong checkout")
        if (target / ".git").is_dir():
            run(git + ["fetch", "origin", self.branch], target)
            run(["git", "checkout", self.branch], target)
            run(git + ["pull", "--ff-only", "origin", self.branch], target)
        else:
            run(git + ["clone", "--branch", self.branch, self.repo_url, str(target)])
        if not (target / "app/local_pipeline.py").is_file():
            raise RuntimeError("Nhánh clone chưa có app/local_pipeline.py; cần push bản full pipeline trước")
        self.root = target.resolve()
        os.chdir(self.root)
        if str(self.root) not in sys.path:
            sys.path.insert(0, str(self.root))
        subprocess.run(["git", "log", "-1", "--oneline"], check=True)
        return self.root

    def install(self):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
        result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "flash-linear-attention"])
        if result.returncode:
            print("flash-linear-attention không khả dụng; dùng implementation torch")

    def check_gpu(self):
        subprocess.run(["nvidia-smi"], check=True)
        import torch
        if not torch.cuda.is_available():
            raise RuntimeError("Cần runtime có CUDA GPU")
        print("torch", torch.__version__, "GPU", torch.cuda.get_device_name(0))

    @staticmethod
    def prepare_output(out):
        if str(out).startswith("/content/drive/"):
            from google.colab import drive
            drive.mount("/content/drive")
        Path(out).mkdir(parents=True, exist_ok=True)

## 1. Clone và cài thư viện
Đặt `GITHUB_TOKEN` trong Colab Secrets, bật Notebook access. Nếu trước đó đã import transformers rồi đổi phiên bản thư viện, restart session và chạy lại từ đầu.

In [ ]:
runtime = ColabRuntime()
ROOT = runtime.checkout()
runtime.install()
runtime.check_gpu()

## 2. Kiểm tra mã và cấu hình pipeline
Các test tích hợp SQL/local, manifest và CLI dùng dữ liệu/backend giả, không tải Qwen và không gọi Gemini. Đây là gate cho pipeline này; bộ test toàn repo còn lỗi có sẵn được ghi trong tài liệu. `OUT` giữ manifest, database, báo cáo và cache. Khi chạy lại, manifest chỉ tái sử dụng nếu khớp nguồn, schema và model; DuckDB chỉ tái sử dụng nếu fingerprint khớp.

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q",
                "tests/test_local_pipeline.py", "tests/test_sql_pipeline.py",
                "tests/test_architecture.py", "tests/test_business_manifest.py",
                "tests/test_manifest_v2.py", "tests/test_main.py"], cwd=ROOT, check=True)

from app.local_pipeline import LocalPipelineConfig, LocalSQLPipeline

OUT = "/content/datamate_sql_out"
# Muốn giữ kết quả qua phiên: OUT = "/content/drive/MyDrive/datamate_sql_out"
runtime.prepare_output(OUT)
config = LocalPipelineConfig(
    dataset_dir=str(ROOT / "dataset"),
    out_dir=OUT,
    model="Qwen/Qwen3.5-9B",
    hf_quant="auto",
    max_model_len=16384,
    manifest_tokens=6144,
    sql_tokens=2048,
    n_samples=3,
    agree_min=2,
)
pipeline = LocalSQLPipeline(config)

## 3. Chuẩn bị toàn bộ dữ liệu trước agent
Bước này nạp Qwen lần đầu, tạo BusinessManifest v2 cho từng business, kiểm định, rồi dựng DuckDB với Raw/Typed/Semantic và catalog. Manifest đầu ra ở `OUT/manifests/<split>/`, database ở `OUT/databases/<split>/<business>/current/data.duckdb`. Không ghi manifest vào dataset gốc.

Chạy lại ô này để tiếp tục sau gián đoạn; các batch manifest hợp lệ đã lưu được dùng lại. Không chuyển sang agent khi bước này còn báo lỗi. Nếu prompt manifest vượt context, thông báo sẽ chỉ rõ; tăng `max_model_len` trong giới hạn VRAM và khởi tạo lại pipeline sau khi restart session.

In [ ]:
preparation = pipeline.prepare_datasets()
validation = pipeline.validate_databases()

## 4. Smoke test: 3 câu
Agent chỉ dùng database đã chuẩn bị và còn hợp lệ. Qwen chuẩn hóa câu hỏi, sinh JSON chứa SQL, nhận lỗi DuckDB để sửa và áp dụng đồng thuận. Xem lỗi sinh/SQL và VRAM trước khi chạy toàn bộ train.

In [ ]:
_, _, smoke_scores, _, smoke_logs = pipeline.smoke_test()
for row in smoke_logs:
    print(row.get("id"), row.get("type"), row.get("sql"), row.get("errors"))

## 5. Đánh giá toàn bộ train
Mặc định xử lý tất cả câu train, không giới hạn thời gian. Đáp án chuẩn chỉ dùng sau suy luận để chấm PA. Khi cần đo holdout: dùng `pipeline.evaluate(subset="holdout", holdout_n=6, per_biz=5)` thay lời gọi phía dưới. Thời gian toàn bộ train cần ước tính từ smoke test; chưa có số đo GPU thực tế trong notebook.

In [ ]:
_, _, train_scores, _, train_logs = pipeline.evaluate()
print(train_scores.status.value_counts())

## 6. Dự đoán toàn bộ public test và kiểm định bài nộp
Không giới hạn số câu/thời gian. Kết quả ở `OUT/public_test/submission.jsonl`, log kèm SQL và lỗi từng câu. Dùng lại cùng đối tượng `pipeline` nên không tải lại Qwen. Trước khi reset runtime, tải file hoặc sao lưu `OUT` nếu đang dùng ổ `/content`.

In [ ]:
submission_path = pipeline.predict_public_test()
print("Bài nộp:", submission_path)
# Tải về máy nếu cần:
# from google.colab import files
# files.download(str(submission_path))